# Playground S6E10 — Predicting Airline Satisfaction

二分类（ROC-AUC）。**Public LB 0.95862 / OOF 0.95914**（LGBM + XGBoost + CatBoost 目标编码集成）。

本 notebook 给出可复现的完整 pipeline：洞察 EDA → 特征工程 → 目标编码 + LGBM 5 折 OOF → 提交。

## 核心洞察（重要）

1. **评分 0 ≠ 最差，而是「服务不适用」**：数字类特征（WiFi / Online boarding / Ease of online booking）评 0 的满意度远高于评 1（如 WiFi 0→0.887 vs 1→0.391）。树模型能自动学到这个非单调关系，无需显式造特征。
2. **最强信号是 `Type of Travel × Customer Type` 交互**（permutation importance 0.0676，是第二名 2 倍多）。
3. 连续特征（Flight Distance / service_std）的 gain 重要性虚高，别被误导。

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

DATA = "data/playground-series-s6e10"   # Kaggle 环境改成 "/kaggle/input/playground-series-s6e10"
TARGET = "satisfaction"
SEED = 42

In [ ]:
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
sub = pd.read_csv(f"{DATA}/sample_submission.csv")
y = train[TARGET].astype(int).values
print(f"train {train.shape}  test {test.shape}")
print(f"正样本率 = {y.mean():.4f}")

## EDA：评分 0 vs 1 的满意度反转

In [ ]:
S = ["Inflight wifi service", "Ease of Online booking", "Online boarding",
     "Food and drink", "Seat comfort", "Inflight entertainment", "Cleanliness"]
for c in S:
    g = train.groupby(c)[TARGET].mean()
    print(f"{c:28s} 0={g.get(0,0):.3f}  1={g.get(1,0):.3f}  5={g.get(5,0):.3f}")

数字类特征（前 3 个）出现明显的「0 > 1」反转 —— 0 表示「没这服务」，不代表差。

## 特征工程

In [ ]:
CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
SERVICE_COLS = [
    "Inflight wifi service", "Departure/Arrival time convenient",
    "Ease of Online booking", "Gate location", "Food and drink",
    "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling",
    "Checkin service", "Cleanliness",
]

def build_features(df):
    df = df.copy()
    df["service_sum"] = df[SERVICE_COLS].sum(axis=1)
    df["service_mean"] = df[SERVICE_COLS].mean(axis=1)
    df["service_std"] = df[SERVICE_COLS].std(axis=1)
    df["total_delay"] = df["Departure Delay in Minutes"] + df["Arrival Delay in Minutes"]
    df["is_delayed"] = (df["total_delay"] > 0).astype(int)
    df["travel_x_class"] = df["Type of Travel"] + "_" + df["Class"]
    df["travel_x_customer"] = df["Type of Travel"] + "_" + df["Customer Type"]
    df["dist_bucket"] = pd.cut(df["Flight Distance"], bins=[0,500,1000,2000,3000,5000,np.inf], labels=False)
    return df

train = build_features(train)
test = build_features(test)

NUM_COLS = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes",
            "service_sum", "service_mean", "service_std", "total_delay", "is_delayed"] + SERVICE_COLS
TE_COLS = CAT_COLS + ["travel_x_class", "travel_x_customer", "dist_bucket"]

for df in (train, test):
    for c in NUM_COLS:
        if df[c].isna().any():
            df[c] = df[c].fillna(0)
    for c in TE_COLS:
        df[c] = df[c].astype("category")

## 目标编码（fold 内 OOF，平滑 m=20）

In [ ]:
def te_encode(X_tr, X_va, X_te, y_tr, cols, m=20.0):
    prior = y_tr.mean()
    out_tr = X_tr[cols].astype(str).copy()
    out_va = X_va[cols].astype(str).copy()
    out_te = X_te[cols].astype(str).copy()
    for c in cols:
        stats = pd.DataFrame({"v": X_tr[c].astype(str), "y": y_tr}).groupby("v")["y"].agg(["mean", "count"])
        enc = (stats["mean"] * stats["count"] + prior * m) / (stats["count"] + m)
        out_tr[c] = out_tr[c].map(enc).fillna(prior)
        out_va[c] = out_va[c].map(enc).fillna(prior)
        out_te[c] = out_te[c].map(enc).fillna(prior)
    return out_tr.astype("float32"), out_va.astype("float32"), out_te.astype("float32")

In [ ]:
N_FOLDS = 5
lgb_params = dict(objective="binary", metric="auc", learning_rate=0.03, num_leaves=127,
                  min_child_samples=50, colsample_bytree=0.7, subsample=0.8, subsample_freq=1,
                  reg_alpha=0.3, reg_lambda=1.0, n_estimators=8000,
                  random_state=SEED, n_jobs=-1, verbosity=-1)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros(len(train))
pred = np.zeros(len(test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y)):
    X_tr, X_va = train.iloc[tr_idx], train.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    te_tr, te_va, te_te = te_encode(X_tr, X_va, test, y_tr, TE_COLS)
    Xt = pd.concat([X_tr[NUM_COLS].reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    Xv = pd.concat([X_va[NUM_COLS].reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    Xte = pd.concat([test[NUM_COLS].reset_index(drop=True), te_te.reset_index(drop=True)], axis=1)
    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(Xt, y_tr, eval_X=Xv, eval_y=y_va, callbacks=[lgb.early_stopping(150, verbose=False)])
    oof[va_idx] = m.predict_proba(Xv)[:, 1]
    pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
    print(f"fold {fold}: best_iter={m.best_iteration_}  val_auc={roc_auc_score(y_va, oof[va_idx]):.5f}")

In [ ]:
print(f"OOF AUC = {roc_auc_score(y, oof):.5f}")
sub[TARGET] = pred
sub.to_csv("submission.csv", index=False)
print("submission.csv saved")

## 结论

- 单 LGBM + 目标编码 ≈ **0.959**；加上 XGBoost + CatBoost 多 seed rank-average blend 后 LB 到 **0.95862**。
- 已证实的硬天花板：GBDT ≈ 0.959、NN ≈ 0.957（MLP 做不好尖锐离散交互分裂）。第一名 0.9614 需要树+NN 之外的手段。
- 关键特征（permutation importance）：`travel_x_customer`（0.0676）、`Inflight wifi service`（0.0304）；`Gender`/延误/service_std 是纯噪声。